# Projet Python : dashboard « Excès de vitesse, radars et accidents en France (2024) »

**Auteur :** GOMES SILVA Victor  ·  Licence Pro Data Mining 2026-2027  ·  Module Python S1

Ce notebook lance un tableau de bord Streamlit destiné à un utilisateur non technique. Il croise :
- les **9,6 millions de vitesses** relevées en 2024 par les voitures radars à conduite externalisée ;
- la **liste des radars fixes** ;
- les **accidents corporels** de 2024 et leur gravité (fichiers BAAC).

Code source : https://github.com/victorgms3/dashboard-radars-2024

**Mode d'emploi :** exécuter les cellules **dans l'ordre**, de haut en bas. Il faut un compte ngrok gratuit
(jeton et domaine, voir `DOCUMENTATION.md`).

## Étape 1 : récupérer le projet et installer les bibliothèques

Par défaut, le projet (code + données préparées, environ 9 Mo) est téléchargé depuis GitHub.
Solution de secours : mettre `SOURCE = "drive"` après avoir déposé le dossier `dashboard_radars` à la racine de *Mon Drive*.

In [ ]:
import os

SOURCE = "github"   # "github" (par défaut) ou "drive" (solution de secours)
DOSSIER = "/content/dashboard-radars-2024"

if SOURCE == "github":
    if os.path.exists(DOSSIER):
        !git -C {DOSSIER} pull -q
    else:
        !git clone -q https://github.com/victorgms3/dashboard-radars-2024.git {DOSSIER}
else:
    from google.colab import drive
    drive.mount('/content/drive')
    !rm -rf {DOSSIER}
    !cp -r /content/drive/MyDrive/dashboard_radars {DOSSIER}

%cd {DOSSIER}
!pip install -q streamlit pyngrok plotly pyarrow scikit-learn

# Facultatif : refaire le nettoyage à partir des 5 fichiers CSV bruts de data.gouv.fr (environ 2 minutes),
# déposés dans le dossier « donnees_brutes_radars » de Mon Drive.
REFAIRE_PREPARATION = False
if REFAIRE_PREPARATION:
    from google.colab import drive
    drive.mount('/content/drive')
    !python preparation_donnees.py /content/drive/MyDrive/donnees_brutes_radars

print(sorted(os.listdir("donnees_preparees")))

## Étape 2 : ouvrir le tunnel ngrok (jeton et domaine personnels)

In [ ]:
###### LANCEMENT DU CODE STREAMLIT via TOKEN & DOMAINS NGROK

from pyngrok import ngrok, conf
from getpass import getpass

ngrok.kill()

token = getpass("Votre token ngrok : ")
domaine = input("Votre domaine ngrok (sans https://) : ").strip()

config = conf.PyngrokConfig(
    auth_token=token,
    config_version="3"
)

public_url = ngrok.connect(
    upstream={"url": "http://localhost:8501"},
    url=f"https://{domaine}",
    bindings=["public"],
    pyngrok_config=config
)

print("Votre dashboard :", public_url.public_url)

## Étape 3 : lancer Streamlit, puis ouvrir le lien affiché à l'étape 2 (cliquer sur « Visit Site »)

In [ ]:
import subprocess
import sys
import time
import requests

# Lancer Streamlit
process = subprocess.Popen([
    sys.executable, "-m", "streamlit",
    "run", "app.py",
    "--server.port", "8501",
    "--server.address", "0.0.0.0",
    "--server.headless", "true"
])

# Attendre que le serveur réponde
for i in range(30):
    try:
        r = requests.get(
            "http://127.0.0.1:8501/_stcore/health",
            timeout=2
        )
        if r.status_code == 200:
            print("✅ Streamlit est opérationnel !")
            break
    except requests.RequestException:
        time.sleep(1)
else:
    print("❌ Streamlit ne répond pas. Vérifie app.py et les logs.")

## Pour arrêter ou relancer

Exécuter la cellule ci-dessous, puis relancer les étapes 2 et 3.

In [ ]:
process.terminate()
ngrok.kill()